# ============================================================
# 1. IMPORTS
# ============================================================


In [52]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error
from sklearn.model_selection import RandomizedSearchCV

# ============================================================
# 2. LOAD & BASIC CLEANING
# ============================================================

In [54]:
flight_df = pd.read_csv(r"raw_bangladesh_data\Flight_Price_Dataset_of_Bangladesh.csv")

df = flight_df.copy()

# Drop irrelevant & leakage columns
df = df.drop(columns=[
    "Source Name",
    "Destination Name",
    "Base Fare (BDT)",
    "Tax & Surcharge (BDT)"
])

# Convert datetime
df["Departure Date & Time"] = pd.to_datetime(df["Departure Date & Time"])
df["Arrival Date & Time"] = pd.to_datetime(df["Arrival Date & Time"])


# ============================================================
# 3. FEATURE ENGINEERING
# ============================================================

In [55]:
df["Month"] = df["Departure Date & Time"].dt.month
df["Day"] = df["Departure Date & Time"].dt.day
df["Weekday"] = df["Departure Date & Time"].dt.weekday
df["Hour"] = df["Departure Date & Time"].dt.hour
df["Is_Weekend"] = df["Weekday"].isin([5,6]).astype(int)


df["Stopovers"] = df["Stopovers"].replace({"Direct": 0})

df["Stopovers"] = df["Stopovers"].str.extract(r'(\d+)')
df["Stopovers"] = df["Stopovers"].fillna(0).astype(int)




df = df.drop(columns=[
    "Departure Date & Time",
    "Arrival Date & Time"
])

# ============================================================
# 4. DEFINE FEATURES & TARGET
# ============================================================

In [56]:
X = df.drop("Total Fare (BDT)", axis=1)
y = df["Total Fare (BDT)"]

numeric_features = [
    "Duration (hrs)",
    "Days Before Departure",
    "Month",
    "Day",
    "Weekday",
    "Hour",
    "Is_Weekend",
    "Stopovers"
]

categorical_features = [
    "Airline",
    "Source",
    "Destination",
    "Aircraft Type",
    "Class",
    "Booking Source",
    "Seasonality"
]

# ============================================================
# 5. PREPROCESSING PIPELINE
# ============================================================

In [57]:
numeric_transformer = StandardScaler()

categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

In [58]:
# Linear Regression Pipeline
lr_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

lr_pipeline.fit(X_train, y_train)

y_pred = lr_pipeline.predict(X_test)

print("R2:", r2_score(y_test, y_pred))
print("MAE:", mean_absolute_error(y_test, y_pred))
print("RMSE:", np.sqrt(mean_squared_error(y_test, y_pred)))


R2: 0.5698563257804181
MAE: 40681.71452848848
RMSE: 53548.879138336306


In [59]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestRegressor(random_state=42))
])

rf_pipeline.fit(X_train, y_train)

y_pred_rf = rf_pipeline.predict(X_test)

print("Random Forest R2:", r2_score(y_test, y_pred_rf))


Random Forest R2: 0.6652460122792132


#### Hyperparameter Tuning

In [62]:
param_grid = {
    "model__n_estimators": [100],
    "model__max_depth": [None, 10]
}

random_search = RandomizedSearchCV(
    rf_pipeline,
    param_grid,
    n_iter=4,
    cv=3,
    scoring="r2",
    n_jobs=-1,
    random_state=42
)

random_search.fit(X_train, y_train)

print("Best Params:", grid.best_params_)
print("Best CV Score:", grid.best_score_)

print(X_train.shape)



c:\Users\MelchizedekTettehNar\OneDrive - AmaliTech gGmbH\Desktop\ML\.venv\Lib\site-packages\sklearn\model_selection\_search.py:324: UserWarning: The total space of parameters 2 is smaller than n_iter=4. Running 2 iterations. For exhaustive searches, use GridSearchCV.
  warnings.warn(


AttributeError: 'GridSearchCV' object has no attribute 'best_params_'